# 0단계 — 01 결과 분석: 패턴 유형별 recall · 수신 계좌 degree 층화
01 노트북의 테스트 점수(`results/HI-Large/basic_features/*_test_scores.parquet`)로 표 두 개를 뽑는다. 02 부터는 같은 표를 자동으로 내고 여기 결과와 나란히 비교한다.

1. **패턴 유형별 recall@1%** — `patterns.parquet` 조인. 임계값은 테스트 전체 상위 1% (알림 예산) 하나로 고정. 패턴 파일에 없는 양성은 UNMATCHED
2. **수신 계좌 in-degree 구간별 F1 / AP** — SALT-GNN 프로토콜. 학습·선택은 그대로 두고 테스트 예측만 받는 계좌의 in-degree 구간으로 나눠서 본다.
   in-degree 는 테스트 구간 끝까지의 거래에서 센 "받은 거래 수"

01 의 트리 모델은 점수 parquet 을 저장하지 않아서 GNN 두 개만 분석한다 (02 부터는 트리도 저장)

## 0. 작업 준비

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [ ]:
cfg = B.Config(dataset='HI-Large')
RUN_DIR = Path.cwd() / 'results' / cfg.dataset / 'basic_features'      # 01 결과 폴더
sorted(p.name for p in RUN_DIR.iterdir())

## 1. 테스트 구간 라벨 + 수신 계좌 in-degree

In [ ]:
con = L.connect(threads=8, memory='12GB')
test_df = B.test_full(con, cfg.test_window)          # edge_id, dst, y, in_pattern, w
in_degree = B.recipient_indegree(con, cfg)            # node_id → 받은 거래 수 (테스트 끝까지)
con.close()
test_df.shape, in_degree.describe().round(1)

In [ ]:
# 수신 계좌 degree 분포 — 테스트 거래가 어느 구간에 얼마나 있나 (양성 비율 포함)
deg = in_degree.reindex(test_df['dst']).to_numpy()
bins = pd.cut(deg, [0, 4, 9, 19, 49, 99, np.inf], labels=['[1-4]', '[5-9]', '[10-19]', '[20-49]', '[50-99]', '[100+]'])
pd.DataFrame({'bin': bins, 'y': test_df['y']}).groupby('bin', observed=True)['y'].agg(n='size', n_pos='sum', pos_rate='mean').round(5)

## 2. 모델별 표

In [ ]:
pattern_tables, degree_tables = [], []
for name in ['gine_basic', 'gine_eu_basic']:
    path = RUN_DIR / f'{name}_test_scores.parquet'
    if not path.exists():
        print(name, '점수 파일 없음'); continue
    sc = B.load_scores(path)
    assert (sc['edge_id'].to_numpy() == test_df['edge_id'].to_numpy()).all(), '점수와 테스트 라벨 순서 불일치'
    pattern_tables.append(B.pattern_recall(sc).assign(model=name))
    degree_tables.append(B.degree_stratified(sc, test_df['dst'].to_numpy(), in_degree).assign(model=name))

In [ ]:
# 패턴 유형별 recall@1% — 열 = 모델. 어느 패턴이 약한지
pattern_df = pd.concat(pattern_tables)
order = [p for p in B.PATTERN_ORDER + ['ALL'] if p in pattern_df.index]
pattern_df.reset_index().pivot(index='pattern_type', columns='model', values='recall').loc[order]

In [ ]:
# 유형별 양성 수 (어느 패턴이 얼마나 있나)
pattern_df[pattern_df.model == pattern_df.model.iloc[0]][['n']]

In [ ]:
# 수신 계좌 degree 구간별 — 허브 구간에서 떨어지나
degree_df = pd.concat(degree_tables)
BIN_ORDER = ['[1-4]', '[5-9]', '[10-19]', '[20-49]', '[50-99]', '[100+]']
degree_df.reset_index().pivot(index='bin', columns='model', values=['AP', 'F1', 'recall_at_budget']).loc[BIN_ORDER]

In [ ]:
degree_df[degree_df.model == 'gine_basic'][['n', 'n_pos', 'pos_rate', 'AP', 'F1', 'F1_at_0_5', 'recall_at_budget']]

## 3. 저장 — 02 부터 비교 기준

In [ ]:
pattern_df.to_csv(RUN_DIR / 'analysis_pattern.csv', encoding='utf-8-sig')
degree_df.to_csv(RUN_DIR / 'analysis_degree.csv', encoding='utf-8-sig')
RUN_DIR